In [1]:
# Parameters
BATCH_MODE = True


In [2]:
# Parameters
BATCH_MODE = True


# PDF Recipe Generator
## Introduction - receipe maker

**Navigation**: [Index](../README.md)

This notebook covers the main concepts and techniques of receipe maker. The educational objectives include understanding the fundamentals, hands-on practice through exercises, and analysis of results.

**Prerequisites**: basic knowledge of Python and algorithms.

> **Adapted from an EPF student production**: **Dorian & Bastien** (cooking). Universalization: refactor [#890](https://github.com/jsboige/CoursIA/pull/890).

This notebook orchestrates a multi-agent collaboration to:
1. Collect user constraints
2. Generate a personalized recipe
3. Produce a styled PDF

**Agents**:  
- `InputCollector` → Collects preferences  
- `RecipeGenerator` → Creates the recipe via LLM  
- `PDFGenerator` → Generates the final document

***

### Installing libraries

**Why this notebook relies on Semantic Kernel rather than a simple API call**: generating a structured recipe (title, listed ingredients, numbered steps, dietary constraints honored) in a single LLM request produces a free-format and often inconsistent result. Semantic Kernel orchestrates **several specialized agents** (criteria collector, recipe generator, PDF producer) that hand off a shared state — it is this *multi-agent* architecture that guarantees the final recipe honors every constraint (vegetarian, no mushrooms, 6 guests). The *import guards* in the next cell make the two optional dependencies explicit (`dotenv`, `semantic_kernel`): if one is missing, the notebook prints it and continues in degraded mode. The other dependencies (`reportlab`, `openai`, `pypdf`) import without a guard: they are required for the generation part, and their absence will show up in the `ImportError` of their cell.


In [3]:
# Import guards - availability flags for external dependencies

try:
    from dotenv import load_dotenv
    DOTENV_AVAILABLE = True
except ImportError:
    DOTENV_AVAILABLE = False
    print(f'  dotenv non disponible - certaines fonctionnalites seront limitees')

try:
    import semantic_kernel
    SEMANTIC_KERNEL_AVAILABLE = True
except ImportError:
    SEMANTIC_KERNEL_AVAILABLE = False
    print(f'  semantic_kernel non disponible - certaines fonctionnalites seront limitees')


## Global State & Configuration

**Why a shared global state is indispensable in multi-agent**: each agent (collector, generator, PDF producer) handles **part** of the problem but must see the **whole** set of constraints. The `RecipeState` class is that shared state — it carries the diet, the excluded ingredients, the number of guests, and the recipe under construction. Without global state, each agent would have to re-pass every constraint in its prompt (redundant, costly in tokens, prone to omissions); with global state, an agent reads `state.diet` and acts accordingly. It is the classic-AI *blackboard* pattern modernized: agents do not talk to each other directly, they communicate **through the state** — which decouples the agents and lets you add one (a nutritionist agent) without modifying the others.


In [4]:
# Etat partage du workflow : les agents (Chef, Critique...) lisent et ecrivent
# ces champs au fil du dialogue ; `ready_to_generate` bascule quand le Chef
# estime que les informations sont completes, `pdf_path` porte la sortie finale.
class RecipeState:
    def __init__(self):
        self.diet: str = ""
        self.excluded_ingredients: list[str] = []
        self.guests: int = 4
        self.ingredients: list[str] = []
        self.steps: list[str] = []
        self.cooking_time: float = 0.0
        self.ready_to_generate: bool = False
        self.pdf_path: str = ""

print("Classe RecipeState définie")


Classe RecipeState définie


### Exercise 1: Enrich the shared state with culinary preferences

The `RecipeState` class stores the diet, excluded ingredients, and number of guests. The goal is to enrich it with new fields: **difficulty level** (beginner, intermediate, expert), **maximum preparation time**, and **preferred cuisine** (French, Italian, Asian, etc.).

**Objective**: extend `RecipeState` with these new attributes and create the corresponding update methods.

**Hints**:
- `# Step 1`: Add the attributes `difficulty`, `max_prep_time`, and `cuisine_type` in `__init__`
- `# Step 2`: Create the methods `set_difficulty`, `set_max_prep_time`, and `set_cuisine`
- `# Hint`: follow the existing pattern of the other attributes (type hints + default value)

In [5]:
class ExtendedRecipeState(RecipeState):
    # TODO etudiant : enrichir l'etat avec les nouvelles preferences
    
    def __init__(self):
        super().__init__()
        self.difficulty: str = ""         # Etape 1 : debutant / intermediaire / expert
        self.max_prep_time: float = 0.0   # Etape 1 : temps max en minutes
        self.cuisine_type: str = ""       # Etape 1 : type de cuisine preferee
    
    def set_difficulty(self, level: str) -> str:
        result = None  # TODO etudiant : valider et assigner
        return result  # TODO etudiant : retourner confirmation
    
    def set_max_prep_time(self, minutes: float) -> str:
        result = None  # TODO etudiant : valider et assigner
        return result  # TODO etudiant : retourner confirmation
    
    def set_cuisine(self, cuisine: str) -> str:
        result = None  # TODO etudiant : assigner
        return result  # TODO etudiant : retourner confirmation

print("Exercice a completer : ExtendedRecipeState")

Exercice a completer : ExtendedRecipeState


### Agent plugins and state modification functions

**Why agents modify state through functions (plugins) rather than by generating free text**: if an agent wrote « diet = vegetarian » in prose, the other agents would have to *parse* that sentence to extract the constraint — fragile and error-prone. The plugins expose **typed functions** (`set_diet`, `exclude_ingredient`, `set_guests`) that the kernel calls in a structured way: the constraint lands directly in the right field of `RecipeState`, with no parsing ambiguity. This is the gap between an agent that *describes* an action (« I note that the user is vegetarian ») and an agent that *executes* an action (`state.diet = vegetarien`) — the latter is reproducible and verifiable, the former depends on the reader's interpretation.


In [6]:
import textwrap
from semantic_kernel.functions import kernel_function
from reportlab.pdfgen import canvas

class InputCollectorPlugin:
    def __init__(self, state: RecipeState):
        self.state = state
    
    @kernel_function(name="set_diet", description="Définit le régime alimentaire")
    def set_diet(self, diet: str) -> str:
        self.state.diet = diet
        return f"Régime {diet} enregistré"

    @kernel_function(name="exclude_ingredient", description="Ajoute un ingrédient à exclure")
    def exclude_ingredient(self, ingredient: str) -> str:
        self.state.excluded_ingredients.append(ingredient)
        return f"Ingrédient {ingredient} exclu"

    @kernel_function(name="set_guests", description="Définit le nombre de convives")
    def set_guests(self, guests: int) -> str:
        self.state.guests = guests
        return f"{guests} convives prévus"

class RecipeGeneratorPlugin:
    def __init__(self, state: RecipeState):
        self.state = state

    @kernel_function(name="submit_recipe", description="Valide la recette générée")
    def submit_recipe(self, ingredients: list[str], steps: list[str], cooking_time: float) -> str:
        self.state.ingredients = ingredients
        self.state.steps = steps
        self.state.cooking_time = cooking_time
        self.state.ready_to_generate = True
        return "Recette validée et prête pour la génération PDF"

class PDFGeneratorPlugin:
    def __init__(self, state: RecipeState):
        self.state = state
    
    @kernel_function(name="generate_pdf", description="Génère le PDF final")
    def generate_pdf(self, output_path: str) -> str:
        c = canvas.Canvas(output_path)
        PAGE_W, PAGE_H = 595, 842  # A4 en points
        MARGIN_L, MARGIN_R, MARGIN_TOP, MARGIN_BOTTOM = 100, 80, 50, 50
        LINE_H = 15
        MAX_LINE_W = (PAGE_W - MARGIN_L - MARGIN_R)  # 415 pts de large utile
        y = PAGE_H - MARGIN_TOP
        
        def draw_wrapped(text, x, y, font_size=10, indent=0):
            """Dessine text enveloppe sur MAX_LINE_W avec retour a la ligne."""
            c.setFont("Helvetica", font_size)
            avg_char_w = font_size * 0.5  # approximation ReportLab Helvetica
            chars_per_line = int(MAX_LINE_W / avg_char_w) - indent
            for line in textwrap.wrap(text, width=chars_per_line) or [""]:
                if y < MARGIN_BOTTOM + LINE_H:
                    c.showPage()  # pagination -- nouvelle page si debordement
                    c.setFont("Helvetica", font_size)
                    y = PAGE_H - MARGIN_TOP
                c.drawString(x + indent, y, line)
                y -= LINE_H
            return y
        
        c.setFont("Helvetica-Bold", 14)
        c.drawString(MARGIN_L, y, "Recette personnalisée")
        y -= 25
        c.setFont("Helvetica", 10)
        y = draw_wrapped(f"Pour {self.state.guests} personnes", MARGIN_L, y)
        y = draw_wrapped(f"Régime: {self.state.diet}", MARGIN_L, y)
        y -= 5
        c.setFont("Helvetica-Bold", 11)
        c.drawString(MARGIN_L, y, "Ingrédients:")
        y -= LINE_H
        c.setFont("Helvetica", 10)
        for ing in (self.state.ingredients or []):
            y = draw_wrapped(f"- {ing}", MARGIN_L, y, indent=15)
        y -= 5
        c.setFont("Helvetica-Bold", 11)
        c.drawString(MARGIN_L, y, "Étapes:")
        y -= LINE_H
        c.setFont("Helvetica", 10)
        for i, step in enumerate(self.state.steps or [], start=1):
            y = draw_wrapped(f"{i}. {step}", MARGIN_L, y, indent=15)
        y -= 5
        c.setFont("Helvetica-Oblique", 10)
        y = draw_wrapped(f"Temps de cuisson: {self.state.cooking_time} min", MARGIN_L, y)
        c.save()
        self.state.pdf_path = output_path
        return f"PDF généré : {output_path}"

print("Imports Semantic Kernel OK")


Imports Semantic Kernel OK


### Exercise 2: Nutrition Estimation Plugin

The current plugins handle information collection and recipe generation, but no nutritional information is provided. The goal is to add a `NutritionPlugin` that estimates the calories and macronutrients of a recipe.

**Objective**: create a `NutritionPlugin` class with an `estimate_nutrition` method annotated with `@kernel_function` that returns an estimate of calories per serving.

**Hints**:
- `# Step 1`: Define an approximate dictionary of calories per basic ingredient
- `# Step 2`: Calculate the sum of calories and divide by the number of diners
- `# Hint`: the dictionary can contain average values (e.g., chicken = 165 kcal/100g, rice = 130 kcal/100g)

In [7]:
class NutritionPlugin:
    """Plugin d'estimation nutritionnelle pour les recettes."""
    
    def __init__(self, state: RecipeState):
        self.state = state
    
    @kernel_function(name="estimate_nutrition", description="Estime les calories par portion")
    def estimate_nutrition(self, ingredients: str) -> str:
        # TODO etudiant : implementer l'estimation nutritionnelle
        # Etape 1 : dictionnaire de reference (ingredient -> kcal/100g)
        CALORIES_REF = {}  # TODO etudiant : completer le dictionnaire
        
        # Etape 2 : parser les ingredients et calculer les calories totales
        total_calories = 0  # TODO etudiant : calculer
        calories_par_portion = None  # TODO etudiant : diviser par state.guests
        
        result = None  # TODO etudiant : formater le resultat
        return result  # TODO etudiant : retourner la description nutritionnelle

print("Exercice a completer : NutritionPlugin")

Exercice a completer : NutritionPlugin


### Creating agents

**Why several agents rather than a single « generate a recipe » prompt**: a single agent would have to handle simultaneously the **collection** of criteria, the **generation** of the recipe, and the **production** of the PDF deliverable — three cognitively different roles, each with its own prompt and reasoning system. Splitting into specialized agents (an empathetic collector that asks the right questions, a creative generator that proposes, a PDF producer that finalizes the deliverable) follows the **single responsibility** principle applied to LLMs: each agent has a clear role, a focused prompt, and a measurable success criterion. This is also what makes the system debuggable — if a recipe violates a constraint, you know to interrogate the prompt and the functions of the agent concerned (here, the generator).


In [8]:
import os
from dotenv import load_dotenv
from openai import AsyncOpenAI
from semantic_kernel import Kernel
from semantic_kernel.agents import ChatCompletionAgent
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
CHAT_MODEL = os.getenv("OPENAI_CHAT_MODEL", "gpt-5-mini")


def build_kernel(plugin, plugin_name: str) -> Kernel:
    """Cree un kernel avec un service de chat OpenAI et le plugin fourni.

    Sans add_service, l'agent leve "No service found" des le premier appel LLM.

    Note gpt-5 / function-tools : SK 1.42 refuse la valeur `none` pour
    `reasoning_effort` (Literal["low","medium","high"] uniquement), mais
    OpenAI l'accepte pour /v1/chat/completions et le requiert pour les
    function tools (400 : "Function tools with reasoning_effort are not
    supported for gpt-5-mini"). On force donc la valeur via `default_query`
    sur le client AsyncOpenAI, ce qui ajoute le parametre a TOUTES les
    requetes HTTP du service (chat, completions, etc.) sans validation
    pydantic cote SK. `async_client=` est le seul point d'injection reconnu
    par OpenAIChatCompletion.__init__ pour un client custom (le kwarg
    `client=` n'apparait pas dans la signature).
    """
    client = AsyncOpenAI(
        api_key=OPENAI_API_KEY,
        default_query={"reasoning_effort": "none"},
    )
    kernel = Kernel()
    kernel.add_service(
        OpenAIChatCompletion(
            service_id="openai",
            ai_model_id=CHAT_MODEL,
            async_client=client,
        )
    )
    kernel.add_plugin(plugin, plugin_name)
    return kernel


shared_state = RecipeState()

# InputCollector
input_kernel = build_kernel(InputCollectorPlugin(shared_state), "input_plugin")
input_agent = ChatCompletionAgent(
    kernel=input_kernel,
    name="InputCollector",
    instructions="""Collectez les informations utilisateur de maniere structuree.
    - Demandez d'abord le regime alimentaire
    - Puis les ingredients a exclure
    - Enfin le nombre de convives"""
)

# RecipeGenerator avec son plugin
recipe_kernel = build_kernel(RecipeGeneratorPlugin(shared_state), "recipe_plugin")
recipe_agent = ChatCompletionAgent(
    kernel=recipe_kernel,
    name="RecipeGenerator",
    instructions="""Generez des recettes en respectant les contraintes.
    - Convertir les ingredients en liste Python
    - Structurer les etapes de preparation
    - Calculer le temps de cuisson"""
)

# PDFGenerator
pdf_kernel = build_kernel(PDFGeneratorPlugin(shared_state), "pdf_plugin")
pdf_agent = ChatCompletionAgent(
    kernel=pdf_kernel,
    name="PDFGenerator",
    instructions="""Generez un PDF professionnel:
    - Structurez en sections claires
    - Utilisez une mise en page aeree
    - Quand la recette est validee, appelez la fonction generate_pdf avec
      output_path='recette_generee.pdf'"""
)

print("Imports et configuration OK")


Imports et configuration OK


### creating the conversation with termination and/or selection criteria

**Why the conversation needs explicit termination criteria**: without a stopping rule, the agents could loop forever (one agent asks for a clarification, the other answers, the first one asks again…). Termination criteria define **when the conversation stops** — here: as soon as the PDF deliverable is produced (the strategy polls the shared state), or after a maximum number of exchanges. Selection criteria define **which agent speaks next** — this notebook defines none: the default selection gives the floor to the agents in turn, in list order, which the executed conversation shows (`InputCollector` then `RecipeGenerator` then `PDFGenerator`). These two mechanisms turn a potentially chaotic stream of LLM exchanges into an **ordered protocol** with a guaranteed beginning, middle and end.


In [9]:
from semantic_kernel.agents import AgentGroupChat
from pydantic import PrivateAttr

from semantic_kernel.agents.strategies import TerminationStrategy

class ReadyTerminationStrategy(TerminationStrategy):
    _state: RecipeState = PrivateAttr()

    def __init__(self, state: RecipeState, **kwargs):
        super().__init__(**kwargs)
        self._state = state

    async def should_agent_terminate(self, agent, history):
        # Terminer sur le LIVRABLE produit (pdf_path), pas sur la demande
        # de le produire (ready_to_generate) : sinon la conversation
        # s'arrête exactement au tour où PDFGenerator devrait parler.
        return bool(self._state.pdf_path)


group_chat = AgentGroupChat(
    agents=[input_agent, recipe_agent, pdf_agent],
    termination_strategy=ReadyTerminationStrategy(shared_state, maximum_iterations=12),
)

print("Imports agents OK")


Imports agents OK


### Main loop

**Why the main loop orchestrates instead of executing sequentially**: the loop does not stop after one round per agent — as long as the deliverable is not produced (and up to the iteration maximum), the conversation resumes: an agent can take the floor again on the next round to complete whatever is missing. This is the difference between a *pipeline* (a single pass, no return) and a *conversation* (iterative, with feedback). The main loop is the conductor: it reads the state, gives the floor to the next agent (default selection: round-robin), executes the agent, checks the termination criteria, and repeats until convergence. This is what allows the system to **correct** its own errors instead of propagating them.


In [10]:
# Fallback: s'assurer que group_chat est défini
try:
    group_chat
except NameError:
    print("⚠️ group_chat non défini - vérifiez que la cellule précédente a été exécutée")
    group_chat = None

from semantic_kernel.contents import ChatMessageContent, AuthorRole

async def recipe_workflow():
    if group_chat is None:
        print("❌ Impossible d'exécuter: group_chat non initialisé")
        return

    # En mode batch (Papermill/CI), stdin n'existe pas : demande de recette
    # fixée -- même patron que Medical-Chatbot et Fort-Boyard.
    if BATCH_MODE:
        recipe_query = (
            "Je souhaite une recette végétarienne pour 6 personnes, "
            "sans champignons ni produits laitiers."
        )
        print(f"[Batch] Demande de démonstration : {recipe_query}")
    else:
        recipe_query = input("Décrivez votre demande de recette : ")

    # Amorcer la conversation (API SK 1.42 : add_chat_message est asynchrone,
    # l'appeler sans await laisse un coroutine jamais attendue).
    await group_chat.add_chat_message(
        ChatMessageContent(role=AuthorRole.USER, content=recipe_query)
    )

    try:
        async for message in group_chat.invoke():
            print(f"[{message.name}] {message.content}")
    except Exception as e:
        print(f"Erreur lors de la génération: {str(e)}")

    print("\nProcessus terminé")

await recipe_workflow()

[Batch] Demande de démonstration : Je souhaite une recette végétarienne pour 6 personnes, sans champignons ni produits laitiers.


[InputCollector] Merci — j'ai bien enregistré vos préférences :
- Régime : végétarien
- Ingrédients à exclure : champignons, produits laitiers
- Convives : 6

Voici une recette adaptée — végétarienne, sans champignons ni produits laitiers — pour 6 personnes : Gratin de légumes et lentilles à la crème de cajou (sans lait).

Ingrédients
- 400 g de lentilles vertes ou du Puy, rincées
- 2 oignons moyens, émincés
- 3 gousses d'ail, hachées
- 2 carottes, coupées en rondelles
- 1 courgette, coupée en demi-lunes
- 1 poivron rouge, coupé en morceaux
- 400 g de tomates concassées (1 boîte) ou 4 tomates fraîches pelées et concassées
- 200 g d'épinards frais (ou surgelés, bien égouttés)
- 200 g de crème de cajou (voir préparation ci-dessous) ou 200 g de crème végétale non laitière (avoine ou soja) si vous préférez
- 2 cuillères à soupe d'huile d'olive
- 1 cuillère à café de paprika fumé (ou doux)
- 1/2 cuillère à café de cumin moulu
- Sel et poivre au goût
- Herbes fraîches : persil ou coriandre p

[RecipeGenerator] Recette enregistrée et validée.

Je vous fournis ci-dessous la recette au format demandé (ingrédients en liste Python, étapes structurées et calcul du temps de cuisson).

Ingrédients (liste Python)
ingredients = [
    "400 g de lentilles vertes (rincées)",
    "2 oignons moyens (émincés)",
    "3 gousses d'ail (hachées)",
    "2 carottes (en rondelles)",
    "1 courgette (en demi-lunes)",
    "1 poivron rouge (en morceaux)",
    "400 g de tomates concassées (ou 4 tomates fraîches pelées et concassées)",
    "200 g d'épinards frais (ou surgelés, bien égouttés)",
    "200 g de crème de cajou (ou 200 g de crème végétale sans lait : avoine/soja)",
    "2 c. à soupe d'huile d'olive",
    "1 c. à café de paprika fumé (ou doux)",
    "1/2 c. à café de cumin moulu",
    "Sel et poivre au goût",
    "Persil ou coriandre frais (pour servir)",
    "3 c. à soupe de chapelure (optionnelle, vérifier sans lait)",
    "50 g de noix concassées ou pignons (optionnel)",
    "Pour la crè

[PDFGenerator] J'ai généré le PDF "recette_generee.pdf" contenant la recette végétarienne (6 personnes) sans champignons ni produits laitiers. 

Souhaitez-vous que je vous envoie le contenu du PDF ici, que je l'ajuste (format, police, ajout d'images), ou que je crée aussi une liste de courses séparée ?

Processus terminé


**Reading the generation** — the multi-agent system produced a vegetarian recipe (the model varies the recipe at each run — we do not freeze its name in the notebook) for **6 people** that respects the collected constraints: **no mushrooms** and **no dairy products**. The `[InputCollector]` first recorded the three criteria (diet, exclusions, guests), then the `[RecipeGenerator]` proposed a suitable recipe — here we see the two agents collaborating through the shared state.

**What this output demonstrates about the multi-agent system**: the recipe is not a free-form response to “write a vegetarian recipe” — it is the **result of a protocol** in which each constraint was explicitly collected and then propagated to the generator via `RecipeState`. If we changed “6 guests” to “2 guests,” the recipe quantities would adjust because the generator reads `state.guests`. This is the added value of the architecture compared with a monolithic prompt: the constraints are **structured and traceable**, not buried in a sentence. This is also what makes the system reusable for other cases (meal menu, shopping list) by changing the plugins, not by rewriting the prompt.

### Exercise 3: Constraint validation before generation

The current workflow launches generation without checking that the constraints are coherent. The objective is to implement a function `valider_contraintes` that checks the coherence of user preferences before passing them on to the agents.

**Objective**: write a function that detects inconsistencies (for example: vegan diet + request for cheese, negative number of guests, excluded ingredients that are part of the diet).

**Hints**:
- `# Step 1`: Define the validation rules (list of conditions to check)
- `# Step 2`: Return a tuple `(est_valide, liste_erreurs)` to guide the user
- `# Hint`: use if/elif conditions for each rule and accumulate error messages

In [11]:
def valider_contraintes(state: RecipeState) -> tuple[bool, list[str]]:
    # TODO etudiant : implementer la validation des contraintes
    erreurs = []
    
    # Etape 1 : verifier la coherence du regime et des ingredients exclus
    # Etape 2 : verifier le nombre de convives, la duree, etc.
    
    est_valide = len(erreurs) == 0  # TODO etudiant : ajouter les conditions
    return est_valide, erreurs

print("Exercice a completer : validation des contraintes")

Exercice a completer : validation des contraintes


### PDF generation

**Why export the recipe as a PDF**: the PDF is the **final deliverable** — what the user takes away, prints, shares. The entire multi-agent pipeline leads to this document, produced by `PDFGeneratorPlugin.generate_pdf` (`reportlab`, imported with the plugins).

**The orchestration trap lurking at this step**: the conversation is driven by `ReadyTerminationStrategy`, which queries the shared state at the end of every turn. If the strategy terminates on `ready_to_generate` — the flag `RecipeGenerator` sets when it **validates** the recipe — the conversation stops **exactly on the turn where `PDFGenerator` should speak**: the agent is listed in the roster passed to `AgentGroupChat`, but it is structurally unreachable. No exception is raised; the loop ends cleanly, and the failure reads as a rendering-tool problem (PDF library, fonts) when the actual cause is the **stop condition**. The rule applied here: **terminate on the produced deliverable** (`state.pdf_path`), not on the request to produce it.

In [12]:
import os

PDF_PATH = "recette_generee.pdf"

# Filet de sécurité : si l'agent PDFGenerator n'a pas invoqué generate_pdf
# pendant la conversation, on produit le livrable directement via le même
# plugin que celui exposé à l'agent -- la capacité démontrée est identique.
if not shared_state.pdf_path:
    pdf_plugin = PDFGeneratorPlugin(shared_state)
    print(pdf_plugin.generate_pdf(PDF_PATH))

if shared_state.pdf_path and os.path.exists(shared_state.pdf_path):
    taille = os.path.getsize(shared_state.pdf_path)
    print(f"PDF présent sur le disque : {shared_state.pdf_path} ({taille} octets)")
else:
    print("Aucun PDF produit : ni par l'agent, ni par le filet de sécurité.")

PDF présent sur le disque : recette_generee.pdf (4081 octets)


The PDF file **recette_generee.pdf** is confirmed to be present on the disk with a size of a few kilobytes (the size varies depending on the generated recipe), validating that the generation produced a usable file.

In [13]:
import pypdf
import re

# Phase 1 -- extraction : on relit le PDF produit par la cellule precedente
# et on en fait un texte plat comparable a l'etat attendu.
pdf_path = shared_state.pdf_path
reader = pypdf.PdfReader(pdf_path)
pdf_text = chr(10).join(p.extract_text() or "" for p in reader.pages)
print(f"PDF généré avec succès: {pdf_path}")
print(f"Taille: {os.path.getsize(pdf_path)} octets")
print(f"Contenu extrait ({len(pdf_text)} caracteres):")
print(pdf_text)


def _norm(s):
    # Normalisation commune : espaces multiples, casse et bordures ; le texte
    # extrait d'un PDF contient des sauts de ligne au milieu des phrases.
    return re.sub(r"\s+", " ", s).strip().lower()


def _endswith(s, n_words=2):
    words = _norm(s).split()
    return " ".join(words[-n_words:]) if len(words) >= n_words else _norm(s)


def _check_ingredient(ing, pdf_text):
    """Vrai si l'ingredient (quantite + nom) est present dans le PDF.

    Test: chaine normalisee directe, OU variante sans virgule, OU split (nom + quantite)
    tous deux presents. Necessaire car le PDF peut mettre l'ingredient sur plusieurs lignes ou
    avec une virgule differente.
    """
    pdf_n = _norm(pdf_text)
    ing_n = _norm(ing)
    if ing_n in pdf_n:
        return True
    ing_no_comma = ing_n.replace(",", " ")
    if ing_no_comma in pdf_n:
        return True
    parts = [p.strip() for p in ing.split(",")]
    if len(parts) == 2 and _norm(parts[0]) in pdf_n and _norm(parts[1]) in pdf_n:
        return True
    return False


def _check_step(s, pdf_text, n_min_overlap=0.4):
    """Vrai si l'etape est presente dans le PDF avec un overlap de mots >= n_min_overlap
    ET les 2 derniers mots de l'etape sont dans le PDF (signe que la fin n'est pas
    tronquee). Le seul debut [:30] de V4 ne protegeait pas contre la troncature finale.
    """
    pdf_n = _norm(pdf_text)
    s_n = _norm(s)
    if s_n in pdf_n:
        return True
    s_words = s_n.split()
    if not s_words:
        return False
    p_words = set(pdf_n.split())
    overlap = sum(1 for w in s_words if w in p_words) / len(s_words)
    # Garde anti-troncature : la fin de l'etape doit exister telle quelle.
    end_ok = _endswith(s, 2) in pdf_n
    return overlap >= n_min_overlap and end_ok


# Phase 2 -- assertions : chaque champ de l'etat partage doit se retrouver
# dans le PDF ; tout manquement leve une erreur (le carnet ne peut pas finir
# vert sur une recette incomplete).
missing = []
if not shared_state.diet or _norm(shared_state.diet) not in _norm(pdf_text):
    missing.append(f"regime={shared_state.diet!r} absent du PDF")
if not shared_state.ingredients:
    missing.append("ingredients vides")
else:
    missing_ing = [i for i in shared_state.ingredients if not _check_ingredient(i, pdf_text)]
    if missing_ing:
        missing.append(
            f"{len(missing_ing)}/{len(shared_state.ingredients)} ingredients absents: {missing_ing[:3]}"
        )
if not shared_state.steps:
    missing.append("etapes vides")
else:
    missing_step = [s for s in shared_state.steps if not _check_step(s, pdf_text)]
    if missing_step:
        missing.append(
            f"{len(missing_step)}/{len(shared_state.steps)} etapes absentes: {missing_step[0][:50]}"
        )
if missing:
    raise RuntimeError(
        f"PDF incomplet -- {len(missing)} chaine(s) manquante(s) : "
        + " / ".join(missing)
    )
print("OK : PDF exporte la recette complete (regime + tous ingredients + toutes etapes).")


PDF généré avec succès: recette_generee.pdf
Taille: 4081 octets
Contenu extrait (2317 caracteres):
Recette personnalisée
Pour 6 personnes
Régime: végétarien
Ingrédients:
- 400 g de lentilles vertes (rincées)
- 2 oignons moyens (émincés)
- 3 gousses d'ail (hachées)
- 2 carottes (en rondelles)
- 1 courgette (en demi-lunes)
- 1 poivron rouge (en morceaux)
- 400 g de tomates concassées (ou 4 tomates fraîches pelées et
concassées)
- 200 g d'épinards frais (ou surgelés, bien égouttés)
- 200 g de crème de cajou (ou 200 g de crème végétale sans lait :
avoine/soja)
- 2 c. à soupe d'huile d'olive
- 1 c. à café de paprika fumé (ou doux)
- 1/2 c. à café de cumin moulu
- Sel et poivre au goût
- Persil ou coriandre frais (pour servir)
- 3 c. à soupe de chapelure (optionnelle, vérifier sans lait)
- 50 g de noix concassées ou pignons (optionnel)
- Pour la crème de cajou maison : 150 g de noix de cajou crues
(trempées 2 h), 120-150 ml d'eau, 1/2 c. à café de sel
Étapes:
1. Préchauffer le four à 190 °C 

**Reading the PDF deliverable** — the conversation played out the three expected roles: `[InputCollector]` recorded the constraints in the shared state, `[RecipeGenerator]` validated the recipe (`ready_to_generate` flag set), then **`[PDFGenerator]` took its turn** by invoking `generate_pdf` — this is precisely the turn that the original stopping condition (`ready_to_generate`) cut off. The verification cell confirms the file on disk (path + size in bytes), then the next cell displays the success branch with the structured content of the shared state.

**What this example demonstrates**: the termination strategy of an `AgentGroupChat` is not a cosmetic detail — it is what decides which agents **actually exist** in the conversation. A flag set one turn too early makes an agent in the list unreachable with no visible error: the loop ends cleanly, the state remains incomplete, and instinct leads one to blame the external tool. Diagnosing this flaw requires reading the **sequence** of turns (who spoke? who never spoke?), not the exceptions — there are none. The reasoning/execution boundary remains real (the `reportlab` plugin can fail independently of the LLM), but the failure observed before the correction was an **orchestration flaw**, not a rendering flaw.

## Conclusion

This notebook implemented a multi-agent collaboration to generate a personalized recipe and produce a PDF.

**Key points**:

- **Shared state**: the `RecipeState` class serves as shared memory for the three agents. Each plugin reads from and writes to this single instance, which makes it possible to pass the user constraints through to PDF generation.
- **Plugins and `@kernel_function`**: the `@kernel_function` decorator exposes Python methods as tools callable by the LLM (`set_diet`, `submit_recipe`, `generate_pdf`).
- **`AgentGroupChat` orchestration**: the `InputCollector`, `RecipeGenerator`, and `PDFGenerator` agents interact in a group conversation, each with its own kernel and chat service.
- **Termination strategy**: `ReadyTerminationStrategy` queries the shared state (`pdf_path`) to stop the loop as soon as the PDF deliverable is produced — terminating earlier (on `ready_to_generate`) would leave the `PDFGenerator` agent unreachable.
- **Service configuration**: each kernel must receive a service via `add_service` (here `OpenAIChatCompletion`), otherwise invocation fails with `No service found`.

**To go further**:

- Enhance the PDF with a styled layout (sections, images, ingredients table) via `reportlab.platypus`.
- Connect a nutritional API to calculate the recipe's calorie intake.
- Generate an illustration of the dish with an image model and insert it into the document.